In [1]:
import os
import json
import requests
from pathlib import Path
from typing import Optional
from datetime import datetime
from tenacity import retry, stop_after_attempt, wait_exponential

ROOT = Path(r"C:\Users\MAITHILI\Care_Companion")

os.environ["GROQ_API_KEY"] = "***********"
os.environ["USDA_API_KEY"] = "*********"

print("Done")

Done


In [2]:
@retry(stop=stop_after_attempt(3), wait=wait_exponential(min=1, max=5))
def check_drug_interactions(medications: list) -> dict:
    """
    Given a list of medication names, check for known 
    interactions and warnings using OpenFDA.
    
    Returns a dict with warnings per drug.
    """
    results = {}

    for med in medications:
        drug_name = med["name"] if isinstance(med, dict) else med
        
        try:
            # Searching OpenFDA drug label for this medication
            resp = requests.get(
                "https://api.fda.gov/drug/label.json",
                params={
                    "search": f'openfda.generic_name:"{drug_name.lower()}"',
                    "limit":  1
                },
                timeout=10
            )

            if resp.status_code != 200:
                results[drug_name] = {"status": "not_found", "warnings": []}
                continue

            data  = resp.json()
            label = data["results"][0]

            # Extracting the most relevant safety fields
            warnings = {
                "drug_name":         drug_name,
                "status":            "found",
                "boxed_warning":     label.get("boxed_warning",      ["None"])[0][:300]
                                     if label.get("boxed_warning") else "None",
                "warnings":          label.get("warnings",           ["None"])[0][:300]
                                     if label.get("warnings") else "None",
                "drug_interactions": label.get("drug_interactions",  ["None"])[0][:400]
                                     if label.get("drug_interactions") else "None",
                "precautions":       label.get("precautions",        ["None"])[0][:300]
                                     if label.get("precautions") else "None",
            }
            results[drug_name] = warnings

        except Exception as e:
            results[drug_name] = {"status": "error", "error": str(e)}

    return results


#testing it
print("Testing Drug Interaction Checker...")
print("=" * 55)

test_meds = [
    {"name": "Metformin",  "dose": "500mg", "frequency": "twice daily"},
    {"name": "Lisinopril", "dose": "10mg",  "frequency": "once daily"},
]

interaction_results = check_drug_interactions(test_meds)

for drug, info in interaction_results.items():
    print(f"\nDrug: {drug}")
    print(f"  Status:      {info['status']}")
    print(f"  Boxed warn:  {info['boxed_warning'][:80]}...")
    print(f"  Interactions:{info['drug_interactions'][:80]}...")

Testing Drug Interaction Checker...

Drug: Metformin
  Status:      found
  Boxed warn:  WARNING: LACTIC ACIDOSIS WARNING: LACTIC ACIDOSIS See full prescribing informati...
  Interactions:7 DRUG INTERACTIONS Table 4 presents clinically significant drug interactions wi...

Drug: Lisinopril
  Status:      found
  Boxed warn:  WARNING: FETAL TOXICITY When pregnancy is detected, discontinue lisinopril and h...
  Interactions:Drug Interactions Lisinopril Hypotension - Patients on Diuretic Therapy: Patient...


In [3]:
def get_medication_cost(drug_name: str, insurance: str = "none") -> dict:
    """
    Find generic alternatives and estimated costs for a medication.
    Uses RxNorm to find generics, then estimates costs based on 
    known pricing data for common diabetes medications.
    """
    
    #Getting RxCUI from RxNorm
    try:
        resp = requests.get(
            "https://rxnav.nlm.nih.gov/REST/rxcui.json",
            params={"name": drug_name},
            timeout=10
        )
        rxcui = resp.json()["idGroup"].get("rxnormId", [None])[0]
    except Exception:
        rxcui = None

    #Finding related generic drugs via RxNorm
    generics = []
    if rxcui:
        try:
            resp = requests.get(
                f"https://rxnav.nlm.nih.gov/REST/rxcui/{rxcui}/related.json",
                params={"tty": "IN"},  # IN = ingredient (generic level)
                timeout=10
            )
            data = resp.json()
            concepts = (data.get("relatedGroup", {})
                           .get("conceptGroup", []))
            for group in concepts:
                for prop in group.get("conceptProperties", []):
                    generics.append(prop.get("name", ""))
        except Exception:
            pass

    #Known pricing database for common diabetes drugs
    KNOWN_PRICES = {
        "metformin":        {"brand": "Glucophage",  "brand_price": 45,  "generic_price": 4},
        "glipizide":        {"brand": "Glucotrol",   "brand_price": 38,  "generic_price": 9},
        "sitagliptin":      {"brand": "Januvia",     "brand_price": 550, "generic_price": 45},
        "lisinopril":       {"brand": "Zestril",     "brand_price": 65,  "generic_price": 8},
        "insulin glargine": {"brand": "Lantus",      "brand_price": 340, "generic_price": 98},
        "atorvastatin":     {"brand": "Lipitor",     "brand_price": 120, "generic_price": 12},
    }

    drug_key    = drug_name.lower()
    price_info  = KNOWN_PRICES.get(drug_key, None)

    if price_info:
        savings = price_info["brand_price"] - price_info["generic_price"]
        return {
            "drug_name":      drug_name,
            "rxcui":          rxcui,
            "brand_name":     price_info["brand"],
            "brand_price":    f"${price_info['brand_price']}/month",
            "generic_price":  f"${price_info['generic_price']}/month",
            "monthly_savings": f"${savings}",
            "annual_savings":  f"${savings * 12}",
            "generics_found": generics[:3],
            "recommendation": (
                f"Ask your pharmacist for generic {drug_name} "
                f"instead of {price_info['brand']}. "
                f"You could save ${savings}/month (${savings*12}/year)."
            )
        }
    else:
        return {
            "drug_name":      drug_name,
            "rxcui":          rxcui,
            "brand_name":     "Unknown",
            "generic_price":  "Check GoodRx.com for current pricing",
            "generics_found": generics[:3],
            "recommendation": f"Ask your pharmacist about generic alternatives for {drug_name}."
        }


#testing it
print("Testing Medication Cost Finder...")
print("=" * 55)

test_drugs = ["Metformin", "Sitagliptin", "Insulin Glargine"]

for drug in test_drugs:
    result = get_medication_cost(drug)
    print(f"\n{drug}:")
    print(f"  Brand:          {result.get('brand_name')} — {result.get('brand_price', 'N/A')}")
    print(f"  Generic:        {result.get('generic_price')}")
    print(f"  Annual savings: {result.get('annual_savings', 'N/A')}")
    print(f"  Tip: {result.get('recommendation', '')[:80]}...")

Testing Medication Cost Finder...

Metformin:
  Brand:          Glucophage — $45/month
  Generic:        $4/month
  Annual savings: $492
  Tip: Ask your pharmacist for generic Metformin instead of Glucophage. You could save ...

Sitagliptin:
  Brand:          Januvia — $550/month
  Generic:        $45/month
  Annual savings: $6060
  Tip: Ask your pharmacist for generic Sitagliptin instead of Januvia. You could save $...

Insulin Glargine:
  Brand:          Lantus — $340/month
  Generic:        $98/month
  Annual savings: $2904
  Tip: Ask your pharmacist for generic Insulin Glargine instead of Lantus. You could sa...


In [4]:
def lookup_nutrition(food_name: str, patient_conditions: list = None) -> dict:
    """
    Look up nutritional info for a food and flag anything
    relevant for diabetes management.
    
    Returns nutrients + a diabetes-specific assessment.
    """
    if patient_conditions is None:
        patient_conditions = ["Type 2 Diabetes"]

    try:
        resp = requests.get(
            "https://api.nal.usda.gov/fdc/v1/foods/search",
            params={
                "query":    food_name,
                "pageSize": 1,
                "api_key":  os.environ.get("USDA_API_KEY", "DEMO_KEY")
            },
            timeout=15
        )
        resp.raise_for_status()
        foods = resp.json().get("foods", [])

        if not foods:
            return {"food": food_name, "status": "not_found"}

        food  = foods[0]
        nutrients_raw = {n["nutrientName"]: n["value"] 
                         for n in food.get("foodNutrients", [])}

        # Extracting the key nutrients for diabetes management
        nutrients = {
            "calories":         nutrients_raw.get("Energy", 0),
            "carbohydrates_g":  nutrients_raw.get("Carbohydrate, by difference", 0),
            "fiber_g":          nutrients_raw.get("Fiber, total dietary", 0),
            "sugars_g":         nutrients_raw.get("Total Sugars", 0),
            "protein_g":        nutrients_raw.get("Protein", 0),
            "fat_g":            nutrients_raw.get("Total lipid (fat)", 0),
            "sodium_mg":        nutrients_raw.get("Sodium, Na", 0),
        }

        
        net_carbs = max(0, nutrients["carbohydrates_g"] - nutrients["fiber_g"])

        # Diabetes assessment based on net carbs per 100g
        if net_carbs < 5:
            assessment = "EXCELLENT — very low net carbs, great for blood sugar control"
            flag       = "green"
        elif net_carbs < 15:
            assessment = "GOOD — moderate carbs, eat in controlled portions"
            flag       = "yellow"
        elif net_carbs < 30:
            assessment = "CAUTION — high carbs, limit portion size"
            flag       = "orange"
        else:
            assessment = "AVOID — very high carbs, will spike blood sugar significantly"
            flag       = "red"

        return {
            "food":         food["description"],
            "fdc_id":       food["fdcId"],
            "per_100g":     nutrients,
            "net_carbs_g":  round(net_carbs, 1),
            "flag":         flag,
            "assessment":   assessment,
            "status":       "found"
        }

    except Exception as e:
        return {"food": food_name, "status": "error", "error": str(e)}


#testing it
print("Testing Nutrition Lookup...")
print("=" * 55)

test_foods = ["white rice", "brown rice", "broccoli", "apple", "cauliflower"]

for food in test_foods:
    result = lookup_nutrition(food)
    if result["status"] == "found":
        print(f"\n{food.upper()}")
        print(f"  Net carbs:  {result['net_carbs_g']}g per 100g")
        print(f"  Flag:       {result['flag'].upper()}")
        print(f"  Assessment: {result['assessment']}")
    else:
        print(f"\n{food}: {result['status']}")

Testing Nutrition Lookup...

WHITE RICE
  Net carbs:  81.3g per 100g
  Flag:       RED
  Assessment: AVOID — very high carbs, will spike blood sugar significantly

BROWN RICE
  Net carbs:  72.0g per 100g
  Flag:       RED
  Assessment: AVOID — very high carbs, will spike blood sugar significantly

BROCCOLI
  Net carbs:  1.2g per 100g
  Flag:       GREEN
  Assessment: EXCELLENT — very low net carbs, great for blood sugar control

APPLE
  Net carbs:  11.7g per 100g
  Flag:       YELLOW
  Assessment: GOOD — moderate carbs, eat in controlled portions

CAULIFLOWER
  Net carbs:  25.0g per 100g
  Flag:       ORANGE
  Assessment: CAUTION — high carbs, limit portion size


In [5]:
def get_fda_alerts(medications: list, days_back: int = 180) -> dict:
    """
    Check for recent FDA drug safety alerts, recalls, and 
    label changes for a patient's medications.
    
    This is the proactive monitoring tool — runs at session start.
    """
    alerts = {}

    for med in medications:
        drug_name = med["name"] if isinstance(med, dict) else med
        drug_alerts = []

        #Drug enforcement (recalls)
        try:
            resp = requests.get(
                "https://api.fda.gov/drug/enforcement.json",
                params={
                    "search": f'product_description:"{drug_name}"',
                    "limit":  3
                },
                timeout=10
            )
            if resp.status_code == 200:
                recalls = resp.json().get("results", [])
                for r in recalls:
                    drug_alerts.append({
                        "type":        "RECALL",
                        "severity":    r.get("classification", "Unknown"),
                        "description": r.get("reason_for_recall", "")[:200],
                        "date":        r.get("recall_initiation_date", ""),
                        "status":      r.get("status", "")
                    })
        except Exception:
            pass

        #Adverse event reports (FAERS)
        try:
            resp = requests.get(
                "https://api.fda.gov/drug/event.json",
                params={
                    "search": f'patient.drug.medicinalproduct:"{drug_name}"'
                              f'+AND+serious:1',
                    "limit":  3,
                    "count":  "patient.reaction.reactionmeddrapt.exact"
                },
                timeout=10
            )
            if resp.status_code == 200:
                events = resp.json().get("results", [])[:3]
                if events:
                    top_reactions = [e["term"] for e in events]
                    drug_alerts.append({
                        "type":        "ADVERSE_EVENTS",
                        "severity":    "INFO",
                        "description": f"Most reported serious reactions: "
                                       f"{', '.join(top_reactions)}",
                        "date":        datetime.now().strftime("%Y-%m-%d"),
                        "status":      "Active monitoring"
                    })
        except Exception:
            pass

        alerts[drug_name] = {
            "alerts_found": len(drug_alerts),
            "alerts":       drug_alerts,
            "checked_at":   datetime.now().isoformat(),
            "summary": (
                f"No active alerts found for {drug_name}."
                if not drug_alerts
                else f"{len(drug_alerts)} alert(s) found for {drug_name}. Review recommended."
            )
        }

    return alerts


#testing it
print("Testing FDA Alert Monitor...")
print("=" * 55)

test_meds = [
    {"name": "Metformin",  "dose": "500mg"},
    {"name": "Lisinopril", "dose": "10mg"},
]

fda_results = get_fda_alerts(test_meds)

for drug, info in fda_results.items():
    print(f"\n{drug}:")
    print(f"  Summary: {info['summary']}")
    for alert in info["alerts"]:
        print(f"  [{alert['type']}] {alert['description'][:80]}...")

Testing FDA Alert Monitor...

Metformin:
  Summary: 3 alert(s) found for Metformin. Review recommended.
  [RECALL] CGMP Deviations: Intermittent exposure to temperature excursion during storage....
  [RECALL] Failed Moisture Limits: Out of specification for water content...
  [RECALL] CGMP Deviations: FDA analysis detected N-Nitrosodimethylamine (NDMA) impurity ab...

Lisinopril:
  Summary: 3 alert(s) found for Lisinopril. Review recommended.
  [RECALL] Labeling: Incorrect or Missing Package Insert...
  [RECALL] Presence of Foreign Tablets: Potential of stray tablet(s) of Amlodipine Besylate...
  [RECALL] CGMP Deviations: finished products manufactured using active pharmaceutical ingr...


In [6]:
def search_pubmed(query: str, max_results: int = 3) -> list:
    """
    Search PubMed for recent research papers relevant to 
    a patient's question or condition.
    
    Returns a list of paper summaries the agent can reference.
    """
    papers = []

    try:
        #Search for paper IDs
        search_resp = requests.get(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
            params={
                "db":      "pubmed",
                "term":    query,
                "retmax":  max_results,
                "retmode": "json",
                "sort":    "relevance"
            },
            timeout=10
        )
        ids = search_resp.json()["esearchresult"]["idlist"]

        if not ids:
            return []

        #Fetch summaries for those IDs
        summary_resp = requests.get(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi",
            params={
                "db":      "pubmed",
                "id":      ",".join(ids),
                "retmode": "json"
            },
            timeout=10
        )
        summaries = summary_resp.json().get("result", {})

        for pmid in ids:
            paper = summaries.get(pmid, {})
            if paper:
                papers.append({
                    "pmid":     pmid,
                    "title":    paper.get("title", "No title"),
                    "authors":  [a["name"] for a in paper.get("authors", [])[:2]],
                    "journal":  paper.get("fulljournalname", ""),
                    "year":     paper.get("pubdate", "")[:4],
                    "url":      f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
                })

    except Exception as e:
        return [{"error": str(e)}]

    return papers


print("Testing PubMed Research Fetcher...")
print("=" * 55)

test_queries = [
    "metformin type 2 diabetes diet",
    "white rice glycemic index diabetes",
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    papers = search_pubmed(query, max_results=2)
    for p in papers:
        print(f"  [{p.get('year')}] {p.get('title', '')[:70]}...")
        print(f"           {p.get('url')}")

Testing PubMed Research Fetcher...

Query: 'metformin type 2 diabetes diet'
  [2002] Reduction in the incidence of type 2 diabetes with lifestyle intervent...
           https://pubmed.ncbi.nlm.nih.gov/11832527/
  [2024] A 5:2 Intermittent Fasting Meal Replacement Diet and Glycemic Control ...
           https://pubmed.ncbi.nlm.nih.gov/38904963/

Query: 'white rice glycemic index diabetes'


In [7]:
TOOL_REGISTRY = {
    "check_drug_interactions": {
        "function":    check_drug_interactions,
        "description": "Check FDA warnings and interactions for a list of medications",
        "input":       "list of medication dicts with name/dose/frequency",
    },
    "get_medication_cost": {
        "function":    get_medication_cost,
        "description": "Find generic alternatives and cost savings for a medication",
        "input":       "drug name as string",
    },
    "lookup_nutrition": {
        "function":    lookup_nutrition,
        "description": "Get nutritional info and diabetes assessment for a food",
        "input":       "food name as string",
    },
    "get_fda_alerts": {
        "function":    get_fda_alerts,
        "description": "Check for recent FDA recalls and adverse event alerts",
        "input":       "list of medication dicts",
    },
    "search_pubmed": {
        "function":    search_pubmed,
        "description": "Search PubMed for research papers on a topic",
        "input":       "search query string",
    },
}

print("Tool Registry:")
print("=" * 55)
for name, info in TOOL_REGISTRY.items():
    print(f"\n  {name}")
    print(f"    {info['description']}")

Tool Registry:

  check_drug_interactions
    Check FDA warnings and interactions for a list of medications

  get_medication_cost
    Find generic alternatives and cost savings for a medication

  lookup_nutrition
    Get nutritional info and diabetes assessment for a food

  get_fda_alerts
    Check for recent FDA recalls and adverse event alerts

  search_pubmed
    Search PubMed for research papers on a topic


In [8]:
#full integration test
import json


from sqlalchemy import create_engine
from sqlalchemy.orm import Session

DB_PATH = ROOT / "memory" / "sqlite" / "carecompanion.db"

#load function
def quick_load(patient_id):
    with open(ROOT / "evaluation" / "test_patients.json", 
              "r", encoding="utf-8") as f:
        patients = json.load(f)
    return next((p for p in patients if p["patient_id"] == patient_id), None)

sarah = quick_load("p001")
print(f"Running all tools for: {sarah['name']}, age {sarah['age']}")
print(f"Medications: {[m['name'] for m in sarah['medications']]}")
print("=" * 55)

#Drug interactions
print("\n[TOOL 1] Drug Interactions")
interactions = check_drug_interactions(sarah["medications"])
for drug, info in interactions.items():
    print(f"  {drug}: {info['status']}")

#Medication costs
print("\n[TOOL 2] Medication Costs")
for med in sarah["medications"]:
    cost = get_medication_cost(med["name"])
    print(f"  {med['name']}: generic={cost.get('generic_price')} "
          f"savings={cost.get('annual_savings', 'N/A')}/yr")

#Nutrition
print("\n[TOOL 3] Nutrition Check")
foods = ["white rice", "broccoli", "oatmeal"]
for food in foods:
    result = lookup_nutrition(food)
    if result["status"] == "found":
        print(f"  {food}: {result['flag'].upper()} "
              f"({result['net_carbs_g']}g net carbs)")

#FDA alerts
print("\n[TOOL 4] FDA Alerts")
alerts = get_fda_alerts(sarah["medications"])
for drug, info in alerts.items():
    print(f"  {drug}: {info['summary'][:60]}")

#PubMed
print("\n[TOOL 5] Research")
papers = search_pubmed("metformin diet type 2 diabetes", max_results=2)
for p in papers:
    print(f"  [{p.get('year')}] {p.get('title','')[:60]}...")

print("all working!.")

Running all tools for: Sarah, age 50
Medications: ['Metformin']

[TOOL 1] Drug Interactions
  Metformin: found

[TOOL 2] Medication Costs
  Metformin: generic=$4/month savings=$492/yr

[TOOL 3] Nutrition Check
  white rice: RED (81.3g net carbs)
  broccoli: GREEN (1.2g net carbs)
  oatmeal: RED (57.8g net carbs)

[TOOL 4] FDA Alerts
  Metformin: 3 alert(s) found for Metformin. Review recommended.

[TOOL 5] Research
  [2002] Reduction in the incidence of type 2 diabetes with lifestyle...
  [2024] A 5:2 Intermittent Fasting Meal Replacement Diet and Glycemi...
all working!.


In [10]:
checks = [
    ("Drug interaction tool", len(check_drug_interactions(
        [{"name": "Metformin"}])) > 0),
    ("Medication cost tool",  "generic_price" in get_medication_cost("Metformin")),
    ("Nutrition tool",        lookup_nutrition("broccoli")["status"] == "found"),
    ("FDA alert tool",        len(get_fda_alerts(
        [{"name": "Metformin"}])) > 0),
    ("PubMed tool",           len(search_pubmed("diabetes", 1)) > 0),
    ("Tool registry built",   len(TOOL_REGISTRY) == 5),
]

print("=" * 45)
print("checking to see if all tools are working")
all_done = True
for label, status in checks:
    icon = "DONE" if status else "FIX THIS"
    if not status:
        all_done = False
    print(f"  [{icon:8s}] {label}")

print()
if all_done:
    print("All tools working.")
else:
    print("fix it!")

checking to see if all tools are working
  [DONE    ] Drug interaction tool
  [DONE    ] Medication cost tool
  [DONE    ] Nutrition tool
  [DONE    ] FDA alert tool
  [DONE    ] PubMed tool
  [DONE    ] Tool registry built

All tools working.
